In [5]:
import nltk
from nltk import CFG, PCFG
from nltk.parse import RecursiveDescentParser, EarleyChartParser, ViterbiParser

# ============================================================
# TASK 10: NLP PARSING TECHNIQUES
# CFG, TOP-DOWN PARSING, EARLEY PARSING AND PCFG
# ============================================================

# 1. DEFINE A CONTEXT-FREE GRAMMAR (CFG) WITHOUT LEFT-RECURSION
# ------------------------------------------------------------

grammar = CFG.fromstring("""
S -> NP VP
S -> WH AUX NP VP
S -> AUX NP VP

NP -> NP_base PP
NP -> NP_base

NP_base -> Det N
NP_base -> Det Adj N
NP_base -> Pronoun
NP_base -> ProperNoun

VP -> V
VP -> V NP
VP -> V NP PP
VP -> V PP
VP -> Aux V
VP -> Aux V NP
VP -> V S

PP -> Prep NP

WH -> 'what' | 'who'
AUX -> 'did' | 'does' | 'do' | 'is' | 'are'

Det -> 'the' | 'a' | 'an' | 'this' | 'that'
Adj -> 'good' | 'new' | 'old' | 'young'

N -> 'student' | 'teacher' | 'book' | 'assignment'
N -> 'apple' | 'dog' | 'cat' | 'letter' | 'report'
N -> 'school' | 'library' | 'park'

Pronoun -> 'he' | 'she' | 'they' | 'we' | 'i'

ProperNoun -> 'john' | 'mary' | 'alice' | 'tom'

V -> 'reads' | 'read' | 'writes' | 'wrote'
V -> 'submitted' | 'submit' | 'likes' | 'liked'
V -> 'eats' | 'ate' | 'saw' | 'sees'
V -> 'runs' | 'ran' | 'teaches' | 'taught'
V -> 'believes' | 'believe'

Aux -> 'can' | 'will' | 'must'

Prep -> 'in' | 'on' | 'at' | 'to' | 'with' | 'from'
""")

print("CFG Grammar created successfully (left-recursion removed).")


# 2. TOKENIZE INPUT SENTENCES (Case-Insensitive)
# ------------------------------------------------------------

def tokenize_sentence(sentence):
    # Lowercase the tokens to match CFG terminal definitions
    return sentence.replace("?", "").replace(".", "").lower().split()


# 3. TOP-DOWN PARSING: RECURSIVE DESCENT
# ------------------------------------------------------------

rd_parser = RecursiveDescentParser(grammar)

def top_down_parse(sentence):
    tokens = tokenize_sentence(sentence)

    print("\nSentence:", sentence)
    print("Top-Down Parsing:")

    try:
        trees = list(rd_parser.parse(tokens))

        if trees:
            print("Result: Grammatical according to CFG")
            for tree in trees[:3]:
                print(tree)
        else:
            print("Result: Not grammatical according to CFG")

    except ValueError:
        print("Result: Not grammatical according to CFG")


# 4. BOTTOM-UP / CHART PARSING: EARLEY PARSER
# ------------------------------------------------------------

earley_parser = EarleyChartParser(grammar)

def earley_parse(sentence):
    tokens = tokenize_sentence(sentence)

    print("\nSentence:", sentence)
    print("Earley Parsing:")

    try:
        trees = list(earley_parser.parse(tokens))

        if trees:
            print("Result: Grammatical according to CFG")
            for tree in trees[:3]:
                print(tree)
        else:
            print("Result: Not grammatical according to CFG")

    except ValueError:
        print("Result: Not grammatical according to CFG")


# 5. SIMPLE SUBJECT-VERB AGREEMENT CHECK
# ------------------------------------------------------------

singular_subjects = {
    "he", "she", "it", "john", "mary", "alice", "tom",
    "the student", "the teacher", "the dog", "the cat"
}

plural_subjects = {
    "they", "we", "students", "teachers"
}

def check_subject_verb_agreement(sentence):
    words = tokenize_sentence(sentence.lower())

    if len(words) < 2:
        return True

    subject = words[0]
    verb = words[1]

    singular_verbs = {
        "reads", "writes", "likes", "eats",
        "sees", "runs", "teaches", "believes"
    }

    base_verbs = {
        "read", "write", "like", "eat",
        "see", "run", "teach", "believe"
    }

    if subject in {"he", "she", "it", "john", "mary", "alice", "tom"}:
        if verb in base_verbs:
            return False

    if subject in {"they", "we"}:
        if verb in singular_verbs:
            return False

    return True


# 6. TEST SENTENCES
# ------------------------------------------------------------

test_sentences = [
    ("The student reads the book.", "grammatical"),
    ("The teacher writes a letter.", "grammatical"),
    ("John likes the apple.", "grammatical"),
    ("Mary saw the dog.", "grammatical"),
    ("The cat eats the apple.", "grammatical"),
    ("The student submitted the assignment.", "grammatical"),
    ("Alice teaches the student.", "grammatical"),
    ("The dog ran in the park.", "grammatical"),
    ("The teacher reads a book.", "grammatical"),
    ("The student writes a report.", "grammatical"),

    ("What did the student submit?", "grammatical"),
    ("Who did Mary see?", "grammatical"),
    ("The student reads the book", "grammatical"),
    ("John likes the dog", "grammatical"),
    ("The teacher teaches the student", "grammatical"),

    ("Student the reads book.", "ungrammatical"),
    ("The reads student book.", "ungrammatical"),
    ("Dog the eats apple.", "ungrammatical"),
    ("The student the book reads.", "ungrammatical"),
    ("Apple the John likes.", "ungrammatical"),

    ("He read the book.", "grammatical"),
    ("She writes a letter.", "grammatical"),
    ("They read the book.", "grammatical"),
    ("We like the teacher.", "grammatical"),
    ("Tom sees the cat.", "grammatical"),

    ("The student in the library reads a book.", "grammatical"),
    ("Mary writes a letter to the teacher.", "grammatical"),
    ("The dog runs in the park.", "grammatical"),
    ("The teacher reads a new book.", "grammatical"),
    ("The young student likes the old book.", "grammatical")
]


# 7. RUN PARSING EXPERIMENTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TOP-DOWN PARSING RESULTS")
print("=" * 60)

for sentence, expected in test_sentences:
    top_down_parse(sentence)


print("\n" + "=" * 60)
print("EARLEY PARSING RESULTS")
print("=" * 60)

for sentence, expected in test_sentences:
    earley_parse(sentence)


# 8. SUBJECT-VERB AGREEMENT TESTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SUBJECT-VERB AGREEMENT CHECK")
print("=" * 60)

agreement_tests = [
    "He reads the book.",
    "They read the book.",
    "She write a letter.",
    "We likes the teacher."
]

for sentence in agreement_tests:
    result = check_subject_verb_agreement(sentence)

    if result:
        print(sentence, "-> Agreement check passed")
    else:
        print(sentence, "-> Agreement check failed")


# 9. PROBABILISTIC CFG (PCFG)
# ------------------------------------------------------------

pcfg_grammar = PCFG.fromstring("""
S -> NP VP [1.0]

NP -> Det N [0.6] | ProperNoun [0.4]

VP -> V NP [0.7] | V [0.3]

Det -> 'the' [0.6] | 'a' [0.4]

N -> 'student' [0.5] | 'book' [0.5]

ProperNoun -> 'John' [0.5] | 'Mary' [0.5]

V -> 'reads' [0.5] | 'likes' [0.5]
""")

viterbi_parser = ViterbiParser(pcfg_grammar)

# Ensure lowercased list of words
pcfg_sentence = "John reads the book".lower().split()

print("\n" + "=" * 60)
print("PCFG VITERBI PARSING")
print("=" * 60)

try:
    pcfg_trees = list(viterbi_parser.parse(pcfg_sentence))

    if pcfg_trees:
        for tree in pcfg_trees:
            print(tree)
            print("Probability:", tree.prob())
    else:
        print("No valid PCFG parse found.")

except ValueError as error:
    print("Parsing error:", error)


# 10. SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)

print("1. CFG grammar was created.")
print("2. Top-Down Recursive Descent parsing was performed.")
print("3. Earley Chart parsing was performed.")
print("4. Subject-verb agreement checks were demonstrated.")
print("5. PCFG parsing using the Viterbi algorithm was performed.")
print("6. Test sentences were used to evaluate parsing.")

print("\nTask 10 completed successfully.")

CFG Grammar created successfully (left-recursion removed).

TOP-DOWN PARSING RESULTS

Sentence: The student reads the book.
Top-Down Parsing:
Result: Grammatical according to CFG
(S
  (NP (NP_base (Det the) (N student)))
  (VP (V reads) (NP (NP_base (Det the) (N book)))))

Sentence: The teacher writes a letter.
Top-Down Parsing:
Result: Grammatical according to CFG
(S
  (NP (NP_base (Det the) (N teacher)))
  (VP (V writes) (NP (NP_base (Det a) (N letter)))))

Sentence: John likes the apple.
Top-Down Parsing:
Result: Grammatical according to CFG
(S
  (NP (NP_base (ProperNoun john)))
  (VP (V likes) (NP (NP_base (Det the) (N apple)))))

Sentence: Mary saw the dog.
Top-Down Parsing:
Result: Grammatical according to CFG
(S
  (NP (NP_base (ProperNoun mary)))
  (VP (V saw) (NP (NP_base (Det the) (N dog)))))

Sentence: The cat eats the apple.
Top-Down Parsing:
Result: Grammatical according to CFG
(S
  (NP (NP_base (Det the) (N cat)))
  (VP (V eats) (NP (NP_base (Det the) (N apple)))))

Senten